In [79]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
import pandas as pd
import astropy
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.table import vstack
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import ttest_rel
from scipy.stats import poisson
from scipy.stats import chi2
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import AutoMinorLocator
import os
from matplotlib.gridspec import GridSpec
import h5py
from pathlib import Path
from IPython.display import display, Math


FIG_DIR = "/raid/scratch/work/alberttg/Masters_project/Plots"

DATA_PRODUCTS_PATH = "/raid/scratch/work/alberttg/Masters_project/Data_products"

DATA_INPUTS_PATH = "/raid/scratch/work/alberttg/Masters_project/Data_inputs"


In [80]:
def read_fit_results():

    alpha_values = []
    log_mstar_values = []
    log_phistar_values = []

    for i in [0, 1, 2, 3, 4, 5]:

        filename = f"results_table_{i}.h5"

        path = os.path.join(DATA_PRODUCTS_PATH, "Schechter_fit_results", filename)

        with h5py.File(path, 'r') as hf:
            fit_results = {key: hf[key][()] for key in hf.keys()}

        alpha_med = fit_results["alpha_median"]
        alpha_upp_err = fit_results["alpha_84"] - alpha_med
        alpha_low_err = alpha_med - fit_results["alpha_16"]

        temp_alpha = np.array([alpha_med, alpha_upp_err, alpha_low_err], dtype=float)
        alpha_values.append(temp_alpha)

        log_mstar_med = fit_results["mstar_median"]
        log_mstar_upp_err = fit_results["mstar_84"] - log_mstar_med
        log_mstar_low_err = log_mstar_med - fit_results["mstar_16"]

        temp_log_mstar = np.array([log_mstar_med, log_mstar_upp_err, log_mstar_low_err], dtype=float)
        log_mstar_values.append(temp_log_mstar)

        log_phistar_med = fit_results["phistar_median"]
        log_phistar_upp_err = fit_results["phistar_84"] - log_phistar_med
        log_phistar_low_err = log_phistar_med - fit_results["phistar_16"]

        temp_log_phistar = np.array([log_phistar_med, log_phistar_upp_err, log_phistar_low_err], dtype=float)
        log_phistar_values.append(temp_log_phistar)

    alpha_values = np.asarray(alpha_values, dtype=object)
    log_mstar_values = np.asarray(log_mstar_values, dtype=object)
    log_phistar_values = np.asarray(log_phistar_values, dtype=object)

    return alpha_values, log_mstar_values, log_phistar_values

In [81]:
def read_smf_table():

    path = os.path.join(DATA_PRODUCTS_PATH, "SMF_binned_table.fits")

    with fits.open(path) as hdu:
        data = hdu[1].data

    table = Table(data)

    for name in table.colnames:
        print(NameError)
        print(table[name])

In [ ]:
def plot_params_vs_z(alpha_values, log_mstar_values, log_phistar_values):

    fig, axes = plt.subplots(3, 1, figsize=(12,16))
    axes = axes.ravel()

    def plot(param_values, y_label, ax):

        param_med = param_values[:,0]
        param_upp_err = param_values[:,1] 
        param_low_err = param_values[:,2]

        redshift = [7.0, 8.0, 9.0, 10.0, 11.0, 12.5]

        ax.set_xlabel(r"Redshift bin centre $z$", fontsize=18)
        ax.set_ylabel(y_label, fontsize=20)

        ax.grid(True)
        ax.tick_params(
                    axis="both",
                    which="major",
                    labelsize=14,
                    length=8,
                    width=1.5,
                    direction="in",
                    top=True,
                    right=True
                )
        ax.xaxis.set_minor_locator(AutoMinorLocator(5))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.tick_params(
            axis="both",
            which="minor",
            length=4,
            width=1,
            direction="in",
            top=True,
            right=True
        )

        ax.errorbar(x=redshift, y=param_med, 
                    yerr=[param_low_err, param_upp_err],
                    color='black', alpha=0.8, ls='None', label = 'our data', capsize=3, elinewidth=1.5, marker='o', markersize=10)

    plot(alpha_values, r"Alpha $\alpha$", axes[0])
    plot(log_mstar_values, r"$\log_{10}(M_\star)$", axes[1])
    plot(log_phistar_values, r"$\log_{10}(\phi_\star)$", axes[2])

    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, "params_vs_z.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [83]:
if __name__ == "__main__":

    alpha_values, log_mstar_values, log_phistar_values = read_fit_results()

    print("alpha: ", alpha_values)

    # read_smf_table()

    plot_params_vs_z(alpha_values, log_mstar_values, log_phistar_values)

alpha:  [[-1.9001486063277286 0.03570018194871283 0.03900862931897664]
 [-2.0198388763083233 0.1291725672266555 0.08407407661987953]
 [-1.6391879230535675 0.14415270854410656 0.1455023868244436]
 [-2.04196811607685 0.45006620429400357 0.1975548982958264]
 [-1.6774017987328107 0.17857418305294392 0.12355228668766882]
 [-1.4278107952839312 0.24147034007996004 0.15199940473142526]]
